# RunOnSave Extension Performance Optimizations

Key improvements implemented to address performance issues and memory leaks.

## Four Critical Improvements

1. **Debouncing Events** - Prevents excessive command execution
2. **Proper Disposal** - Prevents memory leaks
3. **URI Validation** - Ensures URI validity before processing
4. **Command Serialization** - Prevents race conditions

In [ ]:
# Import Required Libraries
import vscode from 'vscode';
import { join } from 'path';
import { exec } from 'child_process';
import { promisify } from 'util';

# For demonstration purposes
print("Libraries imported successfully")

## Debouncing Implementation

```typescript
function debounce<T extends (...args: any[]) => void>(fn: T, delay: number): T {
  let timeout: NodeJS.Timeout | null = null;
  return ((...args: any[]) => {
    if (timeout) {
      clearTimeout(timeout);
    }
    timeout = setTimeout(() => {
      fn(...args);
    }, delay);
  }) as T;
}
```

In [ ]:
# Define validation utilities
def isValidUri(uri):
    """
    Validates if the provided URI is valid and points to a file
    
    Args:
        uri: The URI to validate
        
    Returns:
        bool: True if valid, False otherwise
    """
    if not uri:
        return False
    
    if uri.scheme !== 'file':
        return False
    
    try:
        # Additional validation logic can be added here
        return True
    except:
        return False

# Test the function
test_uri = { scheme: 'file', path: '/path/to/file.js' }
print(f"Is valid URI: {isValidUri(test_uri)}")

## Command Execution Serialization

```typescript
private _stateLock: Promise<void> = Promise.resolve();

private async _withLock<T>(fn: () => Promise<T>): Promise<T> {
  const release = this._stateLock;
  let resolveLock: () => void;
  this._stateLock = new Promise((resolve) => (resolveLock = resolve));

  try {
    await release;
    return await fn();
  } finally {
    resolveLock();
  }
}
```

In [ ]:
# Implement a debounce function for event handling
def debounce(func, wait_ms=300):
    """
    Returns a debounced version of the function
    
    Args:
        func: The function to debounce
        wait_ms: The debounce delay in milliseconds
        
    Returns:
        function: Debounced function
    """
    timer = None
    
    def debounced(*args, **kwargs):
        nonlocal timer
        
        def delayed_function():
            func(*args, **kwargs)
        
        if timer:
            clearTimeout(timer)
        
        timer = setTimeout(delayed_function, wait_ms)
    
    return debounced

# Usage example with VS Code events
def activate(context):
    # Without debouncing - can cause performance issues
    vscode.workspace.onDidSaveTextDocument(handleSaveEvent)
    
    # With debouncing - better performance
    debouncedSaveHandler = debounce(handleSaveEvent, 500)
    vscode.workspace.onDidSaveTextDocument(debouncedSaveHandler)

## Register Commands with Proper Cleanup

Register commands such as `enableRunOnSave` and `disableRunOnSave` and ensure they are disposed of properly using `context.subscriptions`.